# Running PATH on Simulated FRBs

This notebook demonstrates how to run PATH (Probabilistic Association of Transients to Hosts) on simulated FRB populations.

The workflow consists of three main steps:
1. **Generate FRBs**: Create synthetic FRB populations with realistic properties
2. **Assign to Hosts**: Place FRBs in host galaxies with localization errors
3. **Run PATH**: Calculate posterior probabilities for host candidates

This builds on the previous notebooks:
- `Simulate_Generate_FRBs.ipynb`: Generating FRB populations
- `Simulate_Assign_FRBs.ipynb`: Assigning FRBs to host galaxies

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy import units

# Set up plotting style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# Import astropath modules
from astropath.simulations import generate_frbs, assign_frbs_to_hosts
from astropath import run

## Step 1: Generate FRB Population

First, we generate a population of FRBs using `generate_frbs()`. This gives us FRBs with realistic DM, redshift, and host galaxy magnitude distributions for different surveys.

In [ ]:
# Generate 100 CHIME FRBs for demonstration
n_frbs = 100
seed = 42

frbs = generate_frbs(n_frbs, 'CHIME', seed=seed)

print(f"Generated {len(frbs)} FRBs")
print(f"\nFRB properties:")
print(f"  DM range:  {frbs['DM'].min():.0f} - {frbs['DM'].max():.0f} pc/cm³")
print(f"  z range:   {frbs['z'].min():.3f} - {frbs['z'].max():.3f}")
print(f"  m_r range: {frbs['m_r'].min():.2f} - {frbs['m_r'].max():.2f}")

frbs.head()

## Step 2: Load Galaxy Catalog

We need a galaxy catalog to:
1. Assign FRBs to host galaxies
2. Provide candidate hosts for PATH analysis

We'll use the real combined HSC/DECaLs/HECATE catalog if available, otherwise create a mock catalog.

In [ ]:
def create_mock_galaxy_catalog(n_galaxies=50000, seed=42):
    """
    Create a mock galaxy catalog for demonstration.
    """
    np.random.seed(seed)
    
    # Create galaxies across the sky
    ra = np.random.uniform(0., 360., n_galaxies)
    dec = np.random.uniform(-30., 60., n_galaxies)
    
    # Magnitude distribution (peaks around m_r ~ 21-22)
    mag_best = np.random.beta(a=2, b=5, size=n_galaxies) * 10 + 16
    mag_best = np.clip(mag_best, 16., 26.)
    
    # Half-light radii (log-normal distribution)
    half_light = np.random.lognormal(mean=np.log(0.8), sigma=0.6, size=n_galaxies)
    half_light = np.clip(half_light, 0.1, 5.0)
    
    df = pd.DataFrame({
        'ra': ra,
        'dec': dec,
        'mag_best': mag_best,
        'half_light': half_light,
        'ID': np.arange(n_galaxies)
    })
    
    return df


def load_galaxy_catalog(seed=42):
    """
    Load galaxy catalog - tries real catalog first, falls back to mock.
    """
    frb_apath = os.environ.get('FRB_APATH')
    
    if frb_apath is not None:
        catalog_path = Path(frb_apath) / 'combined_HSC_DECaLs_HECATE_galaxies_hecatecut.parquet'
        
        if catalog_path.exists():
            print(f"Loading real galaxy catalog from:")
            print(f"  {catalog_path}")
            galaxies = pd.read_parquet(catalog_path)
            
            required_cols = ['ra', 'dec', 'mag_best', 'half_light', 'ID']
            if all(col in galaxies.columns for col in required_cols):
                print(f"  Loaded {len(galaxies):,} galaxies")
                return galaxies, True
    
    print("Creating mock galaxy catalog...")
    print("  (Set $FRB_APATH to use real HSC/DECaLs/HECATE catalog)")
    galaxies = create_mock_galaxy_catalog(n_galaxies=50000, seed=seed)
    print(f"  Created {len(galaxies):,} mock galaxies")
    return galaxies, False


# Load catalog
galaxies, is_real = load_galaxy_catalog(seed=42)

print(f"\nGalaxy catalog:")
print(f"  N_galaxies: {len(galaxies):,}")
print(f"  Mag range:  {galaxies['mag_best'].min():.2f} - {galaxies['mag_best'].max():.2f}")

## Step 3: Assign FRBs to Host Galaxies

Now we assign FRBs to host galaxies based on their expected host magnitude. The assignment includes:
- True FRB position within the galaxy
- Observed position with localization error
- Localization ellipse parameters (a, b, PA)

In [ ]:
# Define localization error ellipse
# (semi-major axis, semi-minor axis, position angle) in arcsec, arcsec, degrees
localization = (1.0, 0.5, 45.)  # 1" x 0.5" ellipse at 45 deg

print(f"Localization error ellipse:")
print(f"  Semi-major axis (a): {localization[0]}\"")
print(f"  Semi-minor axis (b): {localization[1]}\"")
print(f"  Position angle:      {localization[2]} deg")
print()

# Assign FRBs to hosts
assignments = assign_frbs_to_hosts(
    frb_df=frbs,
    galaxy_catalog=galaxies,
    localization=localization,
    mag_range=(17., 26.),  # Only assign FRBs with hosts in this mag range
    seed=seed
)

print(f"\nAssigned {len(assignments)} FRBs to hosts")
print(f"(Filtered from {len(frbs)} based on magnitude range)")

assignments.head()

### Output columns:

- `ra`, `dec`: Observed FRB coordinates (with localization error)
- `true_ra`, `true_dec`: True FRB coordinates (within galaxy)
- `gal_ID`: Host galaxy ID
- `gal_off`: FRB offset from galaxy center (arcsec)
- `mag`: Host galaxy magnitude
- `half_light`: Host galaxy half-light radius (arcsec)
- `loc_off`: Localization error magnitude (arcsec)
- `a`, `b`, `PA`: Localization ellipse parameters

## Step 4: Run PATH Analysis

Now we run PATH on each assigned FRB. For each FRB, we:
1. Extract nearby galaxies as candidates
2. Build the input dictionary with localization and priors
3. Run PATH to compute posterior probabilities
4. Compare with the true host

In [ ]:
def extract_candidates(frb_row, galaxies, search_radius_arcmin=1.0):
    """
    Extract candidate host galaxies near an FRB position.
    
    Args:
        frb_row: Row from assignments DataFrame
        galaxies: Galaxy catalog DataFrame
        search_radius_arcmin: Search radius in arcmin
        
    Returns:
        astropy.table.Table: Candidate catalog for PATH
    """
    # Get FRB position
    frb_ra = frb_row['ra']
    frb_dec = frb_row['dec']
    
    # Convert search radius to degrees
    search_radius_deg = search_radius_arcmin / 60.
    
    # Simple box cut (fast approximation)
    cos_dec = np.cos(np.radians(frb_dec))
    ra_mask = np.abs(galaxies['ra'] - frb_ra) * cos_dec < search_radius_deg
    dec_mask = np.abs(galaxies['dec'] - frb_dec) < search_radius_deg
    
    nearby = galaxies[ra_mask & dec_mask].copy()
    
    if len(nearby) == 0:
        return None
    
    # Convert to astropy Table for PATH
    catalog = Table()
    catalog['ra'] = nearby['ra'].values
    catalog['dec'] = nearby['dec'].values
    catalog['ang_size'] = nearby['half_light'].values
    catalog['mag'] = nearby['mag_best'].values
    catalog['ID'] = nearby['ID'].values
    
    return catalog

In [ ]:
def run_path_on_frb(frb_row, galaxies, search_radius_arcmin=1.0, verbose=False):
    """
    Run PATH analysis on a single FRB.
    
    Args:
        frb_row: Row from assignments DataFrame
        galaxies: Galaxy catalog DataFrame
        search_radius_arcmin: Search radius for candidates
        verbose: Print detailed output
        
    Returns:
        dict: PATH results including posteriors and true host info
    """
    # Extract candidate galaxies
    catalog = extract_candidates(frb_row, galaxies, search_radius_arcmin)
    
    if catalog is None or len(catalog) == 0:
        return {'success': False, 'reason': 'no_candidates'}
    
    # Build input dictionary
    lparam = {
        'a': frb_row['a'],
        'b': frb_row['b'],
        'theta': frb_row['PA']
    }
    
    idict = run.build_idict(
        ra=frb_row['ra'],
        dec=frb_row['dec'],
        ltype='eellipse',
        lparam=lparam,
        P_O_method='inverse',
        PU=0.1,  # 10% prior on unseen host
        scale=0.5,
        theta_PDF='exp',
        theta_max=6.0
    )
    
    # Run PATH
    try:
        result_candidates, P_Ux, Path, mag_key, cut_catalog, _ = run.run_on_dict(
            idict,
            verbose=verbose,
            catalog=catalog,
            mag_key='mag'
        )
    except Exception as e:
        return {'success': False, 'reason': str(e)}
    
    if result_candidates is None or len(result_candidates) == 0:
        return {'success': False, 'reason': 'path_failed'}
    
    # Find the true host in results
    true_host_id = frb_row['gal_ID']
    
    # Check if true host is in candidates
    if 'ID' in result_candidates.columns:
        true_host_mask = result_candidates['ID'] == true_host_id
        if true_host_mask.any():
            true_host_P_Ox = result_candidates.loc[true_host_mask, 'P_Ox'].values[0]
            true_host_rank = (result_candidates['P_Ox'] > true_host_P_Ox).sum() + 1
        else:
            true_host_P_Ox = 0.0
            true_host_rank = len(result_candidates) + 1
    else:
        true_host_P_Ox = None
        true_host_rank = None
    
    return {
        'success': True,
        'n_candidates': len(result_candidates),
        'P_Ux': P_Ux,
        'top_P_Ox': result_candidates['P_Ox'].iloc[0],
        'true_host_P_Ox': true_host_P_Ox,
        'true_host_rank': true_host_rank,
        'true_host_id': true_host_id,
        'candidates': result_candidates
    }

### Run PATH on a single FRB (example)

In [ ]:
# Run PATH on the first FRB
frb_idx = 0
frb_row = assignments.iloc[frb_idx]

print(f"FRB {frb_idx}:")
print(f"  Position: ({frb_row['ra']:.6f}, {frb_row['dec']:.6f})")
print(f"  True host ID: {frb_row['gal_ID']}")
print(f"  True host mag: {frb_row['mag']:.2f}")
print(f"  Localization: {frb_row['a']:.2f}\" x {frb_row['b']:.2f}\"")
print()

result = run_path_on_frb(frb_row, galaxies, search_radius_arcmin=1.0, verbose=True)

if result['success']:
    print(f"\n" + "="*60)
    print(f"PATH Results:")
    print(f"  N candidates: {result['n_candidates']}")
    print(f"  P(U|x):       {result['P_Ux']:.4f}")
    print(f"  Top P(O|x):   {result['top_P_Ox']:.4f}")
    print(f"  True host P(O|x): {result['true_host_P_Ox']:.4f}")
    print(f"  True host rank:   {result['true_host_rank']}")
else:
    print(f"PATH failed: {result['reason']}")

### Run PATH on all assigned FRBs

In [ ]:
# Run PATH on all assigned FRBs
results = []

print(f"Running PATH on {len(assignments)} FRBs...")
print()

for idx in range(len(assignments)):
    frb_row = assignments.iloc[idx]
    result = run_path_on_frb(frb_row, galaxies, search_radius_arcmin=1.0)
    result['frb_idx'] = idx
    result['frb_id'] = frb_row['FRB_ID']
    result['true_host_mag'] = frb_row['mag']
    result['gal_off'] = frb_row['gal_off']
    result['loc_off'] = frb_row['loc_off']
    results.append(result)
    
    if (idx + 1) % 20 == 0:
        print(f"  Processed {idx + 1}/{len(assignments)} FRBs")

print(f"\nDone! Processed {len(results)} FRBs")

In [ ]:
# Convert results to DataFrame for analysis
results_df = pd.DataFrame([r for r in results if r['success']])

print(f"Successfully ran PATH on {len(results_df)} / {len(results)} FRBs")
print(f"\nSummary statistics:")
print(f"  Mean N candidates:    {results_df['n_candidates'].mean():.1f}")
print(f"  Median true host rank: {results_df['true_host_rank'].median():.1f}")
print(f"  Correct (rank=1):     {(results_df['true_host_rank'] == 1).sum()} ({(results_df['true_host_rank'] == 1).mean()*100:.1f}%)")

## Step 5: Analyze PATH Performance

Let's analyze how well PATH identifies the true hosts.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. True host rank distribution
ax = axes[0, 0]
ranks = results_df['true_host_rank']
max_rank = min(10, ranks.max())
ax.hist(ranks[ranks <= max_rank], bins=np.arange(0.5, max_rank + 1.5, 1), 
        alpha=0.7, edgecolor='black', color='#1f77b4')
ax.set_xlabel('True Host Rank')
ax.set_ylabel('Number of FRBs')
ax.set_title(f'True Host Rank Distribution\n(Rank 1 = {(ranks == 1).sum()}/{len(ranks)} = {(ranks == 1).mean()*100:.1f}%)')
ax.set_xticks(range(1, max_rank + 1))

# 2. True host P(O|x) distribution
ax = axes[0, 1]
ax.hist(results_df['true_host_P_Ox'], bins=20, alpha=0.7, edgecolor='black', color='#2ca02c')
ax.axvline(results_df['true_host_P_Ox'].median(), color='red', linestyle='--', linewidth=2,
           label=f"Median: {results_df['true_host_P_Ox'].median():.3f}")
ax.set_xlabel('True Host P(O|x)')
ax.set_ylabel('Number of FRBs')
ax.set_title('True Host Posterior Probability')
ax.legend()

# 3. P(O|x) vs galaxy offset
ax = axes[1, 0]
scatter = ax.scatter(results_df['gal_off'], results_df['true_host_P_Ox'], 
                     c=results_df['true_host_mag'], cmap='viridis_r', 
                     alpha=0.7, s=40)
ax.set_xlabel('Galaxy Offset (arcsec)')
ax.set_ylabel('True Host P(O|x)')
ax.set_title('Posterior vs Galaxy Offset')
plt.colorbar(scatter, ax=ax, label='Host Magnitude')

# 4. P(O|x) vs localization error
ax = axes[1, 1]
scatter = ax.scatter(results_df['loc_off'], results_df['true_host_P_Ox'],
                     c=results_df['n_candidates'], cmap='plasma',
                     alpha=0.7, s=40)
ax.set_xlabel('Localization Error (arcsec)')
ax.set_ylabel('True Host P(O|x)')
ax.set_title('Posterior vs Localization Error')
plt.colorbar(scatter, ax=ax, label='N Candidates')

plt.tight_layout()
plt.show()

### Cumulative Correct Identification Rate

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Calculate cumulative correct rate by rank
max_rank = 10
ranks = results_df['true_host_rank'].values
cumulative_correct = [(ranks <= r).mean() * 100 for r in range(1, max_rank + 1)]

ax.bar(range(1, max_rank + 1), cumulative_correct, alpha=0.7, edgecolor='black', color='#1f77b4')
ax.plot(range(1, max_rank + 1), cumulative_correct, 'ro-', markersize=8)

# Add value labels
for i, val in enumerate(cumulative_correct):
    ax.text(i + 1, val + 2, f'{val:.1f}%', ha='center', fontsize=10)

ax.set_xlabel('Rank Threshold', fontsize=12)
ax.set_ylabel('Cumulative Correct (%)', fontsize=12)
ax.set_title('Cumulative Correct Host Identification Rate', fontsize=14)
ax.set_xticks(range(1, max_rank + 1))
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Cumulative correct identification:")
for r in [1, 2, 3, 5]:
    if r <= max_rank:
        print(f"  Rank <= {r}: {cumulative_correct[r-1]:.1f}%")

## Step 6: Compare Different Localization Precisions

Let's see how PATH performance varies with localization precision.

In [ ]:
# Define different localization scenarios
localization_scenarios = {
    'Excellent (0.1")': (0.1, 0.1, 0.),
    'Good (0.5")': (0.5, 0.3, 45.),
    'Moderate (2")': (2.0, 1.0, 30.),
    'Poor (5")': (5.0, 3.0, 0.)
}

# Run simulations for each scenario
scenario_results = {}

for scenario_name, loc in localization_scenarios.items():
    print(f"\nRunning {scenario_name}...")
    
    # Assign FRBs with this localization
    assign_scenario = assign_frbs_to_hosts(
        frb_df=frbs,
        galaxy_catalog=galaxies,
        localization=loc,
        mag_range=(17., 26.),
        seed=42
    )
    
    # Run PATH on each
    scenario_path_results = []
    for idx in range(len(assign_scenario)):
        frb_row = assign_scenario.iloc[idx]
        result = run_path_on_frb(frb_row, galaxies, search_radius_arcmin=1.0)
        if result['success']:
            scenario_path_results.append(result)
    
    scenario_results[scenario_name] = pd.DataFrame(scenario_path_results)
    
    # Summary
    df = scenario_results[scenario_name]
    rank1_frac = (df['true_host_rank'] == 1).mean() * 100
    print(f"  Assigned: {len(assign_scenario)}, PATH success: {len(df)}")
    print(f"  Correct (rank=1): {rank1_frac:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# 1. Rank 1 success rate comparison
ax = axes[0]
scenarios = list(scenario_results.keys())
rank1_rates = [(scenario_results[s]['true_host_rank'] == 1).mean() * 100 for s in scenarios]

bars = ax.bar(range(len(scenarios)), rank1_rates, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(scenarios)))
ax.set_xticklabels(scenarios, rotation=15, ha='right')
ax.set_ylabel('Correct Identification Rate (%)')
ax.set_title('PATH Success Rate by Localization Precision')
ax.set_ylim(0, 100)

# Add value labels
for bar, rate in zip(bars, rank1_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
            f'{rate:.1f}%', ha='center', fontsize=11)

# 2. True host P(O|x) distributions
ax = axes[1]
for scenario, color in zip(scenarios, colors):
    df = scenario_results[scenario]
    ax.hist(df['true_host_P_Ox'], bins=20, alpha=0.4, color=color, 
            label=scenario, density=True)

ax.set_xlabel('True Host P(O|x)')
ax.set_ylabel('Density')
ax.set_title('True Host Posterior Distribution by Localization')
ax.legend()

plt.tight_layout()
plt.show()

## Step 7: Effect of Unseen Host Prior (P_U)

The `PU` parameter controls the prior probability that the true host is not in the catalog. Let's see how this affects PATH performance.

In [ ]:
def run_path_with_PU(frb_row, galaxies, PU, search_radius_arcmin=1.0):
    """Run PATH with a specific P_U value."""
    catalog = extract_candidates(frb_row, galaxies, search_radius_arcmin)
    
    if catalog is None or len(catalog) == 0:
        return None
    
    lparam = {
        'a': frb_row['a'],
        'b': frb_row['b'],
        'theta': frb_row['PA']
    }
    
    idict = run.build_idict(
        ra=frb_row['ra'],
        dec=frb_row['dec'],
        ltype='eellipse',
        lparam=lparam,
        P_O_method='inverse',
        PU=PU,
        scale=0.5,
        theta_PDF='exp',
        theta_max=6.0
    )
    
    try:
        result_candidates, P_Ux, Path, mag_key, cut_catalog, _ = run.run_on_dict(
            idict, catalog=catalog, mag_key='mag'
        )
        return {
            'P_Ux': P_Ux,
            'candidates': result_candidates
        }
    except:
        return None

# Test different P_U values on a subset
PU_values = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]
n_test = min(30, len(assignments))

PU_results = {pu: [] for pu in PU_values}

print(f"Testing P_U effect on {n_test} FRBs...")
for idx in range(n_test):
    frb_row = assignments.iloc[idx]
    true_host_id = frb_row['gal_ID']
    
    for pu in PU_values:
        result = run_path_with_PU(frb_row, galaxies, pu)
        if result is not None:
            cands = result['candidates']
            if 'ID' in cands.columns:
                mask = cands['ID'] == true_host_id
                if mask.any():
                    true_P_Ox = cands.loc[mask, 'P_Ox'].values[0]
                else:
                    true_P_Ox = 0.0
            else:
                true_P_Ox = None
            
            PU_results[pu].append({
                'P_Ux': result['P_Ux'],
                'true_host_P_Ox': true_P_Ox
            })

print("Done!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Mean P(U|x) vs prior P_U
ax = axes[0]
mean_PUx = [np.mean([r['P_Ux'] for r in PU_results[pu]]) for pu in PU_values]
ax.plot(PU_values, mean_PUx, 'bo-', markersize=10, linewidth=2)
ax.plot([0, 0.5], [0, 0.5], 'k--', alpha=0.5, label='1:1 line')
ax.set_xlabel('Prior P(U)', fontsize=12)
ax.set_ylabel('Mean Posterior P(U|x)', fontsize=12)
ax.set_title('Effect of Unseen Prior on Posterior')
ax.legend()
ax.grid(alpha=0.3)

# 2. Mean true host P(O|x) vs P_U
ax = axes[1]
mean_true_POx = [np.mean([r['true_host_P_Ox'] for r in PU_results[pu] if r['true_host_P_Ox'] is not None]) 
                 for pu in PU_values]
ax.plot(PU_values, mean_true_POx, 'go-', markersize=10, linewidth=2)
ax.set_xlabel('Prior P(U)', fontsize=12)
ax.set_ylabel('Mean True Host P(O|x)', fontsize=12)
ax.set_title('Effect of Unseen Prior on True Host Posterior')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("P(U) effect summary:")
print(f"{'P_U':<8} {'Mean P(U|x)':<15} {'Mean True Host P(O|x)':<20}")
print("-" * 45)
for pu, pux, pox in zip(PU_values, mean_PUx, mean_true_POx):
    print(f"{pu:<8.2f} {pux:<15.4f} {pox:<20.4f}")

## Summary

This notebook demonstrated the complete workflow for running PATH on simulated FRBs:

1. **Generate FRBs**: Used `generate_frbs()` to create realistic FRB populations
2. **Load Catalog**: Used real or mock galaxy catalog
3. **Assign Hosts**: Used `assign_frbs_to_hosts()` to place FRBs with localization errors
4. **Run PATH**: Used `run.run_on_dict()` to compute posteriors
5. **Analyze Performance**: Evaluated correct identification rates

### Key Findings

- **Localization precision matters**: Better localization leads to higher correct identification rates
- **P_U trade-off**: Higher unseen prior reduces true host posterior but may be more realistic
- **PATH is effective**: Even with moderate localization, PATH correctly identifies most hosts

### Using with Real Data

To use this workflow with real FRB observations:

1. Replace simulated FRBs with real observations
2. Use actual localization parameters from your survey
3. Query appropriate survey catalogs (Pan-STARRS, DECaLS, etc.)
4. Adjust priors based on your expectations